# RI-JK Skeleton 二阶导数优化

In [1]:
from pyscf import gto, scf, lib
import numpy as np
import scipy
from functools import partial
from pyscf.df.grad.rhf import _int3c_wrapper

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = scf.RHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_r_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_r_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_r_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


In [5]:
de_J20 = np.load("nh3_r_hf_decomp.npz")["de_J20"]
de_J11 = np.load("nh3_r_hf_decomp.npz")["de_J11"]
de_J02 = np.load("nh3_r_hf_decomp.npz")["de_J02"]
de_K20 = np.load("nh3_r_hf_decomp.npz")["de_K20"]
de_K11 = np.load("nh3_r_hf_decomp.npz")["de_K11"]
de_K02 = np.load("nh3_r_hf_decomp.npz")["de_K02"]

In [6]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao, nmo = mo_coeff.shape
mocc = mo_coeff[:, mo_occ > 0]
occ_occupation = mo_occ[mo_occ > 0]
nocc = mocc.shape[1]
dm0 = np.dot(mocc, mocc.T) * 2
dme0 = np.einsum('pi,qi,i->pq', mocc, mocc, mo_energy[mo_occ > 0]) * 2
natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()
aux = mf.with_df.auxmol
auxslices = aux.aoslice_by_atom()
naux = aux.nao
mocc_2 = np.einsum("pi,i->pi", mocc, occ_occupation**0.5)

# 原理性分解

下面的分解与 02-*.ipynb 的一些单元非常接近。我们将用这些单元作为验证的参考。

In [7]:
int2c2e = aux.intor("int2c2e")
int2c2e_inv = np.linalg.inv(int2c2e)
int3c2e = _int3c_wrapper(mol, aux, "int3c2e", "s1")()
int3c2e_ip1 = _int3c_wrapper(mol, aux, "int3c2e_ip1", "s1")().reshape([3, nao, nao, naux])
int3c2e_ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip2", "s1")().reshape([3, nao, nao, naux])
int3c2e_ipip1 = _int3c_wrapper(mol, aux, "int3c2e_ipip1", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ipvip1 = _int3c_wrapper(mol, aux, "int3c2e_ipvip1", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ip1ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip1ip2", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ipip2 = _int3c_wrapper(mol, aux, "int3c2e_ipip2", "s1")().reshape([3, 3, nao, nao, naux])
int2c2e_ip1 = aux.intor("int2c2e_ip1")
int2c2e_ipip1 = aux.intor("int2c2e_ipip1").reshape([3, 3, naux, naux])
int2c2e_ip1ip2 = aux.intor("int2c2e_ip1ip2").reshape([3, 3, naux, naux])


### J (basis_2nd)

In [8]:
# (10|0)(0|10)
dbas_J20_1 = np.einsum("tuvP, PQ, sklQ, uv, kl -> tsuk", int3c2e_ip1, int2c2e_inv, int3c2e_ip1, dm0, dm0)
de_J20_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_J20_1[A, B] += 4 * np.einsum("tsuv -> ts", dbas_J20_1[:, :, p0A:p1A, p0B:p1B])

In [9]:
# (11|0)(0|00)
dbas_J20_2 = np.einsum("tsuvP, PQ, klQ, kl -> tsuv", int3c2e_ipvip1, int2c2e_inv, int3c2e, dm0)
de_J20_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_J20_2[A, B] += 2 * np.einsum("tsuv, uv -> ts", dbas_J20_2[:, :, p0A:p1A, p0B:p1B], dm0[p0A:p1A, p0B:p1B])

In [10]:
# (20|0)(0|00)
dbas_J20_3 = np.einsum("tsuvP, PQ, klQ, kl -> tsuv", int3c2e_ipip1, int2c2e_inv, int3c2e, dm0)
de_J20_3 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    de_J20_3[A, A] += 2 * np.einsum("tsuv, uv -> ts", dbas_J20_3[:, :, p0A:p1A], dm0[p0A:p1A])

In [11]:
de_J20_recap = de_J20_1 + de_J20_2 + de_J20_3
assert np.allclose(de_J20_recap, de_J20)

### J (basis_1st ux_1st)

In [12]:
# (10|1)(0|0)(0|00)
dbas_J11_1 = np.einsum("tsuvP, PQ, klQ, uv, kl -> tsuP", int3c2e_ip1ip2, int2c2e_inv, int3c2e, dm0, dm0)
de_J11_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_1[A, B] += 2 * np.einsum("tsuP -> ts", dbas_J11_1[:, :, p0A:p1A, p0B:p1B])
de_J11_1 += de_J11_1.transpose(1, 0, 3, 2)

In [13]:
# (10|0)(0|1)(0|00)
dbas_J11_2 = np.einsum("tuvP, PQ, sQR, RS, klS, uv, kl -> tsuR", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J11_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_2[A, B] += 2 * np.einsum("tsuR -> ts", dbas_J11_2[:, :, p0A:p1A, p0B:p1B])
de_J11_2 += de_J11_2.transpose(1, 0, 3, 2)

In [14]:
# (10|0)(1|0)(0|00)
dbas_J11_3 = np.einsum("tuvP, PQ, sQR, RS, klS, uv, kl -> tsuQ", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J11_3 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_3[A, B] += -2 * np.einsum("tsuQ -> ts", dbas_J11_3[:, :, p0A:p1A, p0B:p1B])
de_J11_3 += de_J11_3.transpose(1, 0, 3, 2)

In [15]:
# (10|0)(0|0)(1|00)
dbas_J11_4 = np.einsum("tuvP, PQ, sklQ, uv, kl -> tsuQ", int3c2e_ip1, int2c2e_inv, int3c2e_ip2, dm0, dm0)
de_J11_4 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_4[A, B] += 2 * np.einsum("tsuQ -> ts", dbas_J11_4[:, :, p0A:p1A, p0B:p1B])
de_J11_4 += de_J11_4.transpose(1, 0, 3, 2)

In [16]:
de_J11_recap = de_J11_1 + de_J11_2 + de_J11_3 + de_J11_4
assert np.allclose(de_J11_recap, de_J11)

### J (aux_2nd)

In [17]:
# (00|2)(0|00)
dbas_J02_1 = np.einsum("tsuvP, PQ, klQ, uv, kl -> tsP", int3c2e_ipip2, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    de_J02_1[A, A] += np.einsum("tsP -> ts", dbas_J02_1[:, :, p0A:p1A])

In [18]:
# (00|0)(2|0)(0|00)
dbas_J02_2 = np.einsum("uvP, PQ, tsQR, RS, klS, uv, kl -> tsQ", int3c2e, int2c2e_inv, int2c2e_ipip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    de_J02_2[A, A] += -1 * np.einsum("tsQ -> ts", dbas_J02_2[:, :, p0A:p1A])
de_J02_2 = de_J02_2

In [19]:
# (00|0)(1|1)(0|00)
dbas_J02_3a = np.einsum("uvP, PQ, tsQR, RS, klS, uv, kl -> tsQR", int3c2e, int2c2e_inv, int2c2e_ip1ip2, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_3a = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_3a[A, B] += -0.5 * np.einsum("tsQR -> ts", dbas_J02_3a[:, :, p0A:p1A, p0B:p1B])
de_J02_3a += de_J02_3a.transpose(1, 0, 3, 2)

In [20]:
# (00|0)(1|0)(0|1)(0|00)
dbas_J02_3b = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, uv, kl -> tsQT", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_3b = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_3b[A, B] += -0.5 * np.einsum("tsQT -> ts", dbas_J02_3b[:, :, p0A:p1A, p0B:p1B])
de_J02_3b += de_J02_3b.transpose(1, 0, 3, 2)

In [21]:
# (00|1)(1|0)(0|00)
dbas_J02_4 = np.einsum("tuvP, PQ, sQR, RS, klS, uv, kl -> tsPQ", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_4 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_4[A, B] += -1 * np.einsum("tsPQ -> ts", dbas_J02_4[:, :, p0A:p1A, p0B:p1B])
de_J02_4 += de_J02_4.transpose(1, 0, 3, 2)

In [22]:
# (00|1)(1|00)
dbas_J02_5 = np.einsum("tuvP, PQ, sklQ, uv, kl -> tsPQ", int3c2e_ip2, int2c2e_inv, int3c2e_ip2, dm0, dm0)
de_J02_5 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_5[A, B] += 0.5 * np.einsum("tsPQ -> ts", dbas_J02_5[:, :, p0A:p1A, p0B:p1B])
de_J02_5 += de_J02_5.transpose(1, 0, 3, 2)

In [23]:
# (00|0)(0|1)(1|0)(0|00)
dbas_J02_6 = np.einsum("uvP, PQ, tRQ, RS, sST, TU, klU, uv, kl -> tsRS", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_6 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_6[A, B] += 0.5 * np.einsum("tsRS -> ts", dbas_J02_6[:, :, p0A:p1A, p0B:p1B])
de_J02_6 += de_J02_6.transpose(1, 0, 3, 2)

In [24]:
# (00|1)(0|1)(0|00)
dbas_J02_7 = np.einsum("tuvP, PQ, sRQ, RS, klS, uv, kl -> tsPR", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_7 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_7[A, B] += -1 * np.einsum("tsPR -> ts", dbas_J02_7[:, :, p0A:p1A, p0B:p1B])
de_J02_7 += de_J02_7.transpose(1, 0, 3, 2)

In [25]:
# (00|0)(1|0)(1|0)(0|00)
dbas_J02_8 = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, uv, kl -> tsRT", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_8 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_8[A, B] += 1 * np.einsum("tsRT -> ts", dbas_J02_8[:, :, p0A:p1A, p0B:p1B])
de_J02_8 += de_J02_8.transpose(1, 0, 3, 2)

In [26]:
de_J02_recap = de_J02_1 + de_J02_2 + de_J02_3a + de_J02_3b + de_J02_4 + de_J02_5 + de_J02_6 + de_J02_7 + de_J02_8
assert np.allclose(de_J02_recap, de_J02, atol=1e-5, rtol=1e-4)

### K (basis_2nd)

In [27]:
# (10|0)(0|10), part a
dbas_K20_1a = np.einsum("tuvP, PQ, sklQ, ui, vj, ki, lj -> tsuk", int3c2e_ip1, int2c2e_inv, int3c2e_ip1, mocc_2, mocc_2, mocc_2, mocc_2)
de_K20_1a = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_K20_1a[A, B] += 2 * np.einsum("tsuk -> ts", dbas_K20_1a[:, :, p0A:p1A, p0B:p1B])

In [28]:
# (10|0)(0|10), part b
dbas_K20_1b = np.einsum("tuvP, PQ, sklQ, ui, vj, kj, li -> tsuk", int3c2e_ip1, int2c2e_inv, int3c2e_ip1, mocc_2, mocc_2, mocc_2, mocc_2)
de_K20_1b = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_K20_1b[A, B] += 2 * np.einsum("tsuk -> ts", dbas_K20_1b[:, :, p0A:p1A, p0B:p1B])

In [29]:
# (11|0)(0|00)
dbas_K20_2 = np.einsum("tsuvP, PQ, klQ, ui, vj, ki, lj -> tsuv", int3c2e_ipvip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K20_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_K20_2[A, B] += 2 * np.einsum("tsuv -> ts", dbas_K20_2[:, :, p0A:p1A, p0B:p1B])

In [30]:
# (20|0)(0|00)
dbas_K20_3 = np.einsum("tsuvP, PQ, klQ, ui, vj, ki, lj -> tsuv", int3c2e_ipip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K20_3 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    de_K20_3[A, A] += 2 * np.einsum("tsuv -> ts", dbas_K20_3[:, :, p0A:p1A])

In [31]:
de_K20_recap = de_K20_1a + de_K20_1b + de_K20_2 + de_K20_3
assert np.allclose(de_K20_recap, de_K20)

### K (basis_1st_aux_1st)

In [32]:
# (10|1)(0|0)(0|00)
dbas_K11_1 = np.einsum("tsuvP, PQ, klQ, vi, li, uj, kj -> tsuP", int3c2e_ip1ip2, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K11_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K11_1[A, B] += 2 * np.einsum("tsuP -> ts", dbas_K11_1[:, :, p0A:p1A, p0B:p1B])
de_K11_1 += de_K11_1.transpose(1, 0, 3, 2)

In [33]:
# (10|0)(0|1)(0|00)
dbas_K11_2 = np.einsum("tuvP, PQ, sQR, RS, klS, ui, vj, ki, lj -> tsuR", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K11_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K11_2[A, B] += 2 * np.einsum("tsuR -> ts", dbas_K11_2[:, :, p0A:p1A, p0B:p1B])
de_K11_2 += de_K11_2.transpose(1, 0, 3, 2)

In [34]:
# (10|0)(1|0)(0|00)
dbas_K11_3 = np.einsum("tuvP, PQ, sQR, RS, klS, ui, vj, ki, lj -> tsuQ", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K11_3 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K11_3[A, B] += -2 * np.einsum("tsuQ -> ts", dbas_K11_3[:, :, p0A:p1A, p0B:p1B])
de_K11_3 += de_K11_3.transpose(1, 0, 3, 2)

In [35]:
# (10|0)(0|0)(1|00)
dbas_K11_4 = np.einsum("tuvP, PQ, sklQ, ui, vj, ki, lj -> tsuQ", int3c2e_ip1, int2c2e_inv, int3c2e_ip2, mocc_2, mocc_2, mocc_2, mocc_2)
de_K11_4 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K11_4[A, B] += 2 * np.einsum("tsuQ -> ts", dbas_K11_4[:, :, p0A:p1A, p0B:p1B])
de_K11_4 += de_K11_4.transpose(1, 0, 3, 2)

In [36]:
de_K11_recap = de_K11_1 + de_K11_2 + de_K11_3 + de_K11_4
assert np.allclose(de_K11_recap, de_K11)

### K (aux_2nd)

In [37]:
# (00|2)(0|00)
dbas_K02_1 = np.einsum("tsuvP, PQ, klQ, ui, vj, ki, lj -> tsP", int3c2e_ipip2, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    de_K02_1[A, A] += np.einsum("tsP -> ts", dbas_K02_1[:, :, p0A:p1A])

In [38]:
# (00|0)(2|0)(0|00)
dbas_K02_2 = np.einsum("uvP, PQ, tsQR, RS, klS, ui, vj, ki, lj -> tsQ", int3c2e, int2c2e_inv, int2c2e_ipip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    de_K02_2[A, A] += -1 * np.einsum("tsQ -> ts", dbas_K02_2[:, :, p0A:p1A])
de_K02_2 = de_K02_2

In [39]:
# (00|0)(1|1)(0|00)
dbas_K02_3a = np.einsum("uvP, PQ, tsQR, RS, klS, ui, vj, ki, lj -> tsQR", int3c2e, int2c2e_inv, int2c2e_ip1ip2, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_3a = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_3a[A, B] += -0.5 * np.einsum("tsQR -> ts", dbas_K02_3a[:, :, p0A:p1A, p0B:p1B])
de_K02_3a += de_K02_3a.transpose(1, 0, 3, 2)

In [40]:
# (00|0)(1|0)(0|1)(0|00)
dbas_K02_3b = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, ui, vj, ki, lj -> tsQT", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_3b = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_3b[A, B] += -0.5 * np.einsum("tsQT -> ts", dbas_K02_3b[:, :, p0A:p1A, p0B:p1B])
de_K02_3b += de_K02_3b.transpose(1, 0, 3, 2)

In [41]:
# (00|1)(1|0)(0|00)
dbas_K02_4 = np.einsum("tuvP, PQ, sQR, RS, klS, ui, vj, ki, lj -> tsPQ", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_4 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_4[A, B] += -1 * np.einsum("tsPQ -> ts", dbas_K02_4[:, :, p0A:p1A, p0B:p1B])
de_K02_4 += de_K02_4.transpose(1, 0, 3, 2)

In [42]:
# (00|1)(1|00)
dbas_K02_5 = np.einsum("tuvP, PQ, sklQ, ui, vj, ki, lj -> tsPQ", int3c2e_ip2, int2c2e_inv, int3c2e_ip2, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_5 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_5[A, B] += 0.5 * np.einsum("tsPQ -> ts", dbas_K02_5[:, :, p0A:p1A, p0B:p1B])
de_K02_5 += de_K02_5.transpose(1, 0, 3, 2)

In [43]:
# (00|0)(0|1)(1|0)(0|00)
dbas_K02_6 = np.einsum("uvP, PQ, tRQ, RS, sST, TU, klU, ui, vj, ki, lj -> tsRS", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_6 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_6[A, B] += 0.5 * np.einsum("tsRS -> ts", dbas_K02_6[:, :, p0A:p1A, p0B:p1B])
de_K02_6 += de_K02_6.transpose(1, 0, 3, 2)

In [44]:
# (00|1)(0|1)(0|00)
dbas_K02_7 = np.einsum("tuvP, PQ, sRQ, RS, klS, ui, vj, ki, lj -> tsPR", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_7 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_7[A, B] += -1 * np.einsum("tsPR -> ts", dbas_K02_7[:, :, p0A:p1A, p0B:p1B])
de_K02_7 += de_K02_7.transpose(1, 0, 3, 2)

In [45]:
# (00|0)(1|0)(1|0)(0|00)
dbas_K02_8 = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, ui, vj, ki, lj -> tsQS", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_8 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_8[A, B] += 1 * np.einsum("tsQS -> ts", dbas_K02_8[:, :, p0A:p1A, p0B:p1B])
de_K02_8 += de_K02_8.transpose(1, 0, 3, 2)

In [46]:
de_K02_recap = de_K02_1 + de_K02_2 + de_K02_3a + de_K02_3b + de_K02_4 + de_K02_5 + de_K02_6 + de_K02_7 + de_K02_8
assert np.allclose(de_K02_recap, de_K02, atol=1e-5, rtol=1e-4)

### J1ao

In [47]:
scr1 = np.einsum("tuvP, PQ, klQ, kl -> tuv", int3c2e_ip1, int2c2e_inv, int3c2e, dm0)

j1ao_aux0 = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = aoslices[A]
    slcA = slice(p0, p1)
    # (10|0)(0|00)
    j1ao_aux0[A, :, slcA, :] -= scr1[:, slcA, :]
    # (01|0)(0|00) (can be symmetrized)
    j1ao_aux0[A, :, :, slcA] -= scr1[:, slcA, :].swapaxes(-1, -2)
    # (00|0)(0|10), (00|0)(0|01)
    scr2 = np.einsum("tklP, PQ, uvQ, kl -> tuv", int3c2e_ip1[:, slcA], int2c2e_inv, int3c2e, dm0[slcA])
    j1ao_aux0[A] -= 2 * scr2

In [48]:
j1ao_aux1 = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    # (00|1)(0|00)
    j1ao_aux1[A] -= np.einsum("tuvP, PQ, klQ, kl -> tuv", int3c2e_ip2[:, :, :, slcA], int2c2e_inv[slcA, :], int3c2e, dm0)
    # (00|0)(1|00)
    j1ao_aux1[A] -= np.einsum("uvP, PQ, tklQ, kl -> tuv", int3c2e, int2c2e_inv[:, slcA], int3c2e_ip2[:, :, :, slcA], dm0)
    # (00|0)(1|0)(0|00)
    j1ao_aux1[A] += np.einsum("uvP, PQ, tQR, RS, klS, kl -> tuv", int3c2e, int2c2e_inv[:, slcA], int2c2e_ip1[:, slcA], int2c2e_inv, int3c2e, dm0)
    # (00|0)(0|1)(0|00)
    j1ao_aux1[A] += np.einsum("uvP, PQ, tRQ, RS, klS, kl -> tuv", int3c2e, int2c2e_inv, int2c2e_ip1[:, slcA], int2c2e_inv[slcA, :], int3c2e, dm0)

### K1ao

In [49]:
scr1 = np.einsum("tuvP, PQ, klQ, vi, li -> tuk", int3c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2)

k1ao_aux0 = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = aoslices[A]
    slcA = slice(p0, p1)
    # (10|0)(0|00)
    k1ao_aux0[A, :, slcA, :] -= scr1[:, slcA, :]
    # (01|0)(0|00)
    k1ao_aux0[A, :, :, slcA] -= scr1[:, slcA, :].swapaxes(-1, -2)
    # (00|0)(0|10), (00|0)(0|01)
    scr2 = np.einsum("tklP, PQ, uvQ, ki, ui -> tlv", int3c2e_ip1[:, slcA], int2c2e_inv, int3c2e, mocc_2[slcA], mocc_2)
    k1ao_aux0[A] -= scr2 + scr2.swapaxes(-1, -2)

In [50]:
# this part of computation is not sutiable using dm-only when einsum
k1ao_aux1 = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    # (00|1)(0|00)
    k1ao_aux1[A] -= np.einsum("tuvP, PQ, klQ, vi, li -> tuk", int3c2e_ip2[:, :, :, slcA], int2c2e_inv[slcA, :], int3c2e, mocc_2, mocc_2)
    # (00|0)(1|00)
    k1ao_aux1[A] -= np.einsum("uvP, PQ, tklQ, vi, li -> tuk", int3c2e, int2c2e_inv[:, slcA], int3c2e_ip2[:, :, :, slcA], mocc_2, mocc_2)
    # (00|0)(1|0)(0|00)
    k1ao_aux1[A] += np.einsum("uvP, PQ, tQR, RS, klS, vi, li -> tuk", int3c2e, int2c2e_inv[:, slcA], int2c2e_ip1[:, slcA], int2c2e_inv, int3c2e, mocc_2, mocc_2)
    # (00|0)(0|1)(0|00)
    k1ao_aux1[A] += np.einsum("uvP, PQ, tRQ, RS, klS, vi, li -> tuk", int3c2e, int2c2e_inv, int2c2e_ip1[:, slcA], int2c2e_inv[slcA, :], int3c2e, mocc_2, mocc_2)

## 基础工具：j2c 转换

PySCF 在处理梯度问题时，会引入函数 `_gen_metric_solve` (调用时以 `solve_j2c` 的形式出现)，以对辅助基指标作转换。

我们这里将要引入一个更复杂的函数。我们引入参数 `left` 与 `flip`。

一些提前的补充说明如下：

- 正常情况下，我们是要用 `overwrite_b = True` 以避免内存开销的。不过一方面这里是草稿，我们先不考虑这个问题；另一方面，在实际程序编写的时候，所有权问题是需要谨慎对待的。
- 这里的函数只保证对二维矩阵有效。比如 2c-2e ERI 导数矩阵的维度是 `tPQ`，这时用 `left = True` 会导致作用的维度是导数分量维度 `t`，而不是辅助基指标维度 `P`。
- 之所以要抽象出一个函数，是因为 j2c 分解不止有一种形式。它还有 eigenvalue 模式、以及 upper cholesky 分解 (REST 的 col-major 倾向) 模式。使用比较统一的函数接口，可以让我们在不同的分解模式下使用同样的代码。

In [ ]:
def gen_solve_by_j2c(int2c):
    int2c_l = scipy.linalg.cholesky(int2c, lower=True)
    def solve_by_j2c(v, flip=False, left=True):
        res = None
        if left and not flip:
            res = scipy.linalg.solve_triangular(int2c_l, v, lower=True)
        elif left and flip:
            res = scipy.linalg.solve_triangular(int2c_l.T, v, lower=False)
        elif not left and not flip:
            res = scipy.linalg.solve_triangular(int2c_l.T, v.T, lower=False).T
        elif not left and flip:
            res = scipy.linalg.solve_triangular(int2c_l, v.T, lower=True).T
        return res
    return solve_by_j2c

In [ ]:
int2c2e_l_inv = scipy.linalg.inv(scipy.linalg.cholesky(int2c2e, lower=True))
solve_by_j2c = gen_solve_by_j2c(int2c2e)


- 默认情况下是 `left = True`, `flip = False`。典型的应用情景是对 3c-2e ERI $g_{P, \mu \nu}$ 转换到分解积分 $Y_{P, \mu \nu}$, $\mathbf{Y} = \mathbf{L}^{-1} \mathbf{g}$ 的情景。这里的 left 是指辅助基 $P$ 在被求解张量 $g_{P, \mu \nu}$ 的左边；flip 是指对于 Cholesky 类型分解 ($\mathbf{L}$ 并非是对称的)，我们不进行转置。

In [ ]:
int3c2e_s2ij = _int3c_wrapper(mol, aux, "int3c2e", "s2ij")()
print(int3c2e_s2ij.shape)
assert np.allclose(solve_by_j2c(int3c2e_s2ij.T), mf.with_df._cderi)
assert np.allclose(int2c2e_l_inv @ int3c2e_s2ij.T, mf.with_df._cderi)

(1225, 131)


作为定义，在 Cholesky 下三角分解的情形下，有下述表达式成立：

In [ ]:
assert np.allclose(solve_by_j2c(int3c2e_s2ij.T, left=True, flip=False), int2c2e_l_inv @ int3c2e_s2ij.T)
assert np.allclose(solve_by_j2c(int3c2e_s2ij.T, left=True, flip=True), int2c2e_l_inv.T @ int3c2e_s2ij.T)
assert np.allclose(solve_by_j2c(int3c2e_s2ij, left=False, flip=False), int3c2e_s2ij @ int2c2e_l_inv)
assert np.allclose(solve_by_j2c(int3c2e_s2ij, left=False, flip=True), int3c2e_s2ij @ int2c2e_l_inv.T)

- `left = False` 的情况会出现在 $(\partial_t P | Q)$ 的两侧都要作转换的情况。有时我们要作类似于下述的转换：$\partial \mathbf{J} \times \mathbf{J}^{-1} \times \partial \mathbf{J}$，为了数值稳定性我们倾向于使用 triangular solve。

In [ ]:
np.allclose(
    solve_by_j2c(int2c2e_ip1[0], left=False, flip=True) @ solve_by_j2c(int2c2e_ip1[1], left=True, flip=False),
    np.einsum("PQ, QR, RS -> PS", int2c2e_ip1[0], int2c2e_inv, int2c2e_ip1[1])
)

True

- `flip = True` 的情形除了上面提到的类相似变换 (左右分别对应不翻转与翻转的两种情形)，另一个常用的情景是连续作两次转换 (即乘以 $\mathbf{J}^{-1}$)。出于数值稳定性，我们有意避免直接存储 $\mathbf{J}^{-1}$ 或分解的逆 ($\mathbf{L}^{-1}$)，而使用矩阵求解实现求逆。但普通矩阵求解通常是 LU 或类似较大开销算法，如果我们已经有 Cholesky 分解，那么就可以用 DTRSM 加快求解速度与降低内存开销。需要调用函数两次确实是麻烦了一些，但这大概是最优做法，也没有必要再抽象出一个新的函数出来。

## 优化尝试

### 补充约定俗成

我们将在后续计算中，按照与 REST 相同的**内存排布顺序** (不是指维度，因此作为 row/col-major 不同的程序，通常维度刚好是相反的)。

在 PySCF 中，指标的顺序是

- 原子指标 `A, B`
- 分量指标 `t, s` (代表 x, y, z)
- 辅助基指标 `P`
- 占据轨道指标 `i`
- 原子轨道指标 `v, u`，**注意这里的顺序是反的，$\nu$ 在前 $\mu$ 在后**

在这里，Hessian 的指标顺序是 `A, B, t, s`，而在 REST 则应该是反过来的 `s, t, B, A`。

需要额外留意的是，对于类似于 ipvip1 的积分 $(\partial_t \mu \partial_s \nu | P)$，在这里的指标顺序是 `t, s, P, v, u`，而在 REST 则是 `u, v, P, s, t`。留意 ipvip1 的 `t` 对应 `u`，`s` 对应 `v`，但这个顺序在张量指标上是对称而不是对应的。

### 电子积分的使用情况

我们首先列举所有出现过的电子积分，以及其对应的导数项。

| 编号 | 导数项 | 电子积分 (3c) | 电子积分 (2c) |
|-----------|---------------------------------|-------------------|--------------|
| 20 - 1    | (10\|0) (0\|10)                 | `ip1`, `ip1`      |              |
| 20 - 2    | (11\|0) (0\|00)                 | `ipvip1`          |              |
| 20 - 3    | (20\|0) (0\|00)                 | `ipip1`           |              |
| 11 - 1    | (10\|1) (0\|0) (0\|00)          | `ip1ip2`          |              |
| 11 - 2    | (10\|0) (0\|1) (0\|00)          | `ip1`             | `ip1`        |
| 11 - 3    | (10\|0) (1\|0) (0\|00)          | `ip1`             | `ip1`        |
| 11 - 4    | (10\|0) (0\|0) (1\|00)          | `ip1`, `ip2`      |              |
| 02 - 1    | (00\|2) (0\|00)                 | `ipip2`           |              |
| 02 - 2    | (00\|0) (2\|0) (0\|00)          |                   | `ipip1`      |
| 02 - 3    | (00\|0) (1\|1) (0\|00)          |                   | `ip1ip2`     |
| 02 - 4    | (00\|1) (1\|0) (0\|00)          | `ip2`             | `ip1`        |
| 02 - 5    | (00\|1) (1\|00)                 | `ip2`, `ip2`      |              |
| 02 - 6    | (00\|0) (0\|1) (1\|0) (0\|00)   |                   | `ip1`, `ip1` |
| 02 - 7    | (00\|1) (0\|1) (0\|00)          | `ip2`             | `ip1`        |
| 02 - 8    | (00\|0) (1\|0) (1\|0) (0\|00)   |                   | `ip1`, `ip1` |
| f1ao_aux0 | (10\|0) (0\|00) and perm        | `ip1`             |              |
| f1ao_aux1 | (00\|1) (0\|00) and perm        | `ip2`             |              |
| f1ao_aux1 | (00\|0) (1\|0) (0\|00) and perm |                   | `ip1`        |

### 优化策略

- **内存优化**。首先要从内存优化角度入手。我们不希望像现在这样，所有的中间张量都被存储在内存中。

  内存控制的总原则是，完全避免 $n_\mathrm{basis}^2 n_\mathrm{aux}$ 级别内存消耗，接受 $n_\mathrm{occ}^2 n_\mathrm{aux}$ 级别的内存消耗。

  大多数项都满足这一条件；但其中有一个特例：`20 - 1` 的 K 积分贡献是唯一麻烦的一项。这一项很可能要求 $3 n_\mathrm{occ} n_\mathrm{basis} n_\mathrm{aux}$ 级别的内存消耗，否则会产生比较严重的额外计算开销。

- **电子积分调用优化**。我们要尽可能减少电子积分的调用次数。尽管 RI-JK 自洽场计算的电子积分耗时不多 (而矩阵乘法耗时更大)，但这单纯是因为我们假设体系不算太大，以至于电子积分全部可以存于内存，从而 $O(N^3)$ 电子积分只计算一次、$O(N^4)$ 矩阵乘法则需要计算多次。但梯度问题则是另一个情景：电子积分与矩阵乘法都只需要计算一次。即使计算复杂度更小，电子积分在梯度计算中的占比更大，耗时也更明显。因此，我们要采用的策略是，算一部分电子积分后，就尽可能蒿完它的羊毛，把相关的项都结算掉，不要留着下一次再算。

- **辅助基转换**。我们将使用类似于 PySCF 的 `solve_j2c` 函数 (由 `_gen_metric_solver` 得到)。但同时，我们也需要允许二次转换，以避免一次对导数张量的、经常是不必要的转换。该转换对于 Cholesky 分解的 J2C 而言需要是 inplace DTRSM 的。在 REST 中，对应的函数是 `get_solved_j3c`；但该函数有可能未来命名为 `solve_by_j2c`，且允许接受 `TensorMut` 类型。